# svm_walk_forward_validation_tuning

SVM validation tuning using the full-history session-aligned dataset.

The notebook builds compact SVM and feature-transformation grids, splits on the full session calendar before removing neutral targets, selects feature/hyperparameter configurations with expanding-window walk-forward validation, calibrates the final decision threshold on the holdout validation period, and evaluates the frozen model on the untouched test split.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)

In [2]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    "dataset_path": PROJECT_ROOT / "data" / "datasets" / "stock_panel_nine_tickers_session_aligned_full_history_adjusted_google_score_raw.csv",
    "excluded_tickers": ["NFLX"],
    "neutral_band": 0.005,
    "test_size": 0.25,
    "validation_fraction_within_pretest": 0.25,
    "min_validation_dates": 20,
    "gap_days": 1,
    "random_state": 42,
    "selection_metric": "selection_score",
    "primary_validation_metric": "balanced_accuracy",
    "walk_forward_folds": 4,
    "walk_forward_validation_dates": 80,
    "walk_forward_min_train_dates": 252,
    "walk_forward_stability_penalty": 0.25,
    "max_features_per_model": 9,
    "tune_decision_threshold": True,
    "threshold_min_quantile": 0.05,
    "threshold_max_quantile": 0.95,
    "threshold_grid_size": 181,
    "bootstrap_iterations": 1000,
    "bootstrap_ci": 0.95,
}

LINEAR_C_VALUES = [0.01, 0.1, 1.0, 10.0]
RBF_C_VALUES = [0.1, 1.0, 10.0]
RBF_GAMMA_VALUES = ["scale", 0.001, 0.01, 0.1]
SVM_CLASS_WEIGHT = "balanced"


def format_grid_value(value: object) -> str:
    if value is None:
        return "none"
    if isinstance(value, float):
        return f"{value:g}".replace(".", "p")
    return str(value).replace(".", "p")


SVM_PARAM_GRID = []
for c_value in LINEAR_C_VALUES:
    SVM_PARAM_GRID.append(
        {
            "param_set": f"linear_C{format_grid_value(c_value)}_{SVM_CLASS_WEIGHT}",
            "kernel": "linear",
            "C": c_value,
            "class_weight": SVM_CLASS_WEIGHT,
        }
    )

for c_value in RBF_C_VALUES:
    for gamma_value in RBF_GAMMA_VALUES:
        SVM_PARAM_GRID.append(
            {
                "param_set": f"rbf_C{format_grid_value(c_value)}_gamma{format_grid_value(gamma_value)}_{SVM_CLASS_WEIGHT}",
                "kernel": "rbf",
                "C": c_value,
                "gamma": gamma_value,
                "class_weight": SVM_CLASS_WEIGHT,
            }
        )

PRICE_FEATURES = [
    "return_1d",
    "return_5d",
    "return_20d",
    "rolling_volatility_20d",
]

VOLUME_FEATURE_OPTIONS = {
    "volume zscore 10d": ["volume_zscore_10d"],
    "volume zscore 20d": ["volume_zscore_20d"],
    "volume zscore 60d": ["volume_zscore_60d"],
    "volume zscore 20d clipped": ["volume_zscore_20d_clip3"],
    "volume log1p zscore 20d": ["volume_log1p_zscore_20d"],
    "volume percentile rank 20d": ["volume_rank_20d"],
}

GDELT_FEATURE_OPTIONS = {
    "GDELT zscore short": ["gdelt_sentiment_zscore_10d", "gdelt_article_count_zscore_6d"],
    "GDELT zscore medium": ["gdelt_sentiment_zscore_20d", "gdelt_article_count_zscore_20d"],
    "GDELT zscore short clipped": ["gdelt_sentiment_zscore_10d_clip3", "gdelt_article_count_zscore_6d_clip3"],
    "GDELT sentiment zscore + log articles": ["gdelt_sentiment_zscore_10d", "gdelt_article_count_log1p_zscore_6d"],
    "GDELT percentile rank 20d": ["gdelt_sentiment_rank_20d", "gdelt_article_count_rank_20d"],
    "GDELT zscore short + missing flag": ["gdelt_sentiment_zscore_10d", "gdelt_article_count_zscore_6d", "gdelt_sentiment_missing"],
}

GDELT_SENTIMENT_FEATURE_OPTIONS = {
    "GDELT sentiment zscore short": ["gdelt_sentiment_zscore_10d"],
    "GDELT sentiment zscore medium": ["gdelt_sentiment_zscore_20d"],
    "GDELT sentiment zscore short clipped": ["gdelt_sentiment_zscore_10d_clip3"],
    "GDELT sentiment percentile rank 20d": ["gdelt_sentiment_rank_20d"],
}

REDDIT_FEATURE_OPTIONS = {
    "Reddit zscore short": ["reddit_sentiment_zscore_6d", "reddit_comment_count_zscore_6d"],
    "Reddit zscore medium": ["reddit_sentiment_zscore_20d", "reddit_comment_count_zscore_20d"],
    "Reddit zscore short clipped": ["reddit_sentiment_zscore_6d_clip3", "reddit_comment_count_zscore_6d_clip3"],
    "Reddit sentiment zscore + log comments": ["reddit_sentiment_zscore_6d", "reddit_comment_count_log1p_zscore_6d"],
    "Reddit percentile rank 20d": ["reddit_sentiment_rank_20d", "reddit_comment_count_rank_20d"],
    "Reddit zscore short + missing flag": ["reddit_sentiment_zscore_6d", "reddit_comment_count_zscore_6d", "reddit_sentiment_missing"],
}


GDELT_ATTENTION_FEATURE_OPTIONS = {
    "GDELT attention zscore short": ["gdelt_article_count_zscore_6d"],
    "GDELT attention zscore medium": ["gdelt_article_count_zscore_20d"],
    "GDELT attention log1p zscore short": ["gdelt_article_count_log1p_zscore_6d"],
    "GDELT attention percentile rank 20d": ["gdelt_article_count_rank_20d"],
}

REDDIT_ATTENTION_FEATURE_OPTIONS = {
    "Reddit attention zscore short": ["reddit_comment_count_zscore_6d"],
    "Reddit attention zscore medium": ["reddit_comment_count_zscore_20d"],
    "Reddit attention log1p zscore short": ["reddit_comment_count_log1p_zscore_6d"],
    "Reddit attention percentile rank 20d": ["reddit_comment_count_rank_20d"],
}

GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS = {
    "Google score attention zscore 10d": ["google_trends_zscore_10d"],
    "Google score attention zscore 20d": ["google_trends_zscore_20d"],
    "Google score attention zscore 60d": ["google_trends_zscore_60d"],
    "Google score attention zscore 20d clipped": ["google_trends_zscore_20d_clip3"],
    "Google score attention percentile rank 20d": ["google_trends_rank_20d"],
}

GOOGLE_TRENDS_FEATURE_OPTIONS = {
    "Google train-median flag": ["google_trends_above_ticker_train_median"],
    "Google zscore 10d": ["google_trends_zscore_10d"],
    "Google zscore 20d": ["google_trends_zscore_20d"],
    "Google zscore 60d": ["google_trends_zscore_60d"],
    "Google zscore 20d clipped": ["google_trends_zscore_20d_clip3"],
    "Google percentile rank 20d": ["google_trends_rank_20d"],
}
DERIVED_FEATURE_COLUMNS = {"google_trends_above_ticker_train_median"}

BASE_VOLUME_OPTION = "volume zscore 20d"
BASELINE_FEATURE_SET = f"Model B - price + volume | {BASE_VOLUME_OPTION}"


def feature_spec(
    feature_set: str,
    feature_family: str,
    *,
    volume_option: str | None = None,
    gdelt_option: str | None = None,
    reddit_option: str | None = None,
    google_option: str | None = None,
) -> dict:
    return {
        "feature_set": feature_set,
        "feature_family": feature_family,
        "volume_option": volume_option,
        "gdelt_option": gdelt_option,
        "reddit_option": reddit_option,
        "google_option": google_option,
    }


FEATURE_SET_SPECS = [feature_spec("Model A - price only", "price only")]

for option_name in VOLUME_FEATURE_OPTIONS:
    FEATURE_SET_SPECS.append(
        feature_spec(
            f"Model B - price + volume | {option_name}",
            "price + volume",
            volume_option=option_name,
        )
    )

for option_name in GDELT_FEATURE_OPTIONS:
    FEATURE_SET_SPECS.append(
        feature_spec(
            f"Model C - price + volume + GDELT | {option_name}",
            "price + volume + GDELT",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option=option_name,
        )
    )

for option_name in REDDIT_FEATURE_OPTIONS:
    FEATURE_SET_SPECS.append(
        feature_spec(
            f"Model E - price + volume + Reddit | {option_name}",
            "price + volume + Reddit",
            volume_option=BASE_VOLUME_OPTION,
            reddit_option=option_name,
        )
    )

for option_name in GOOGLE_TRENDS_FEATURE_OPTIONS:
    FEATURE_SET_SPECS.append(
        feature_spec(
            f"Model G - price + volume + Google | {option_name}",
            "price + volume + Google",
            volume_option=BASE_VOLUME_OPTION,
            google_option=option_name,
        )
    )


for option_name in GDELT_ATTENTION_FEATURE_OPTIONS:
    FEATURE_SET_SPECS.append(
        feature_spec(
            f"Model J - price + volume + GDELT attention | {option_name}",
            "price + volume + GDELT attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option=option_name,
        )
    )

for option_name in REDDIT_ATTENTION_FEATURE_OPTIONS:
    FEATURE_SET_SPECS.append(
        feature_spec(
            f"Model K - price + volume + Reddit attention | {option_name}",
            "price + volume + Reddit attention",
            volume_option=BASE_VOLUME_OPTION,
            reddit_option=option_name,
        )
    )

for option_name in GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS:
    FEATURE_SET_SPECS.append(
        feature_spec(
            f"Model L - price + volume + Google score attention | {option_name}",
            "price + volume + Google score attention",
            volume_option=BASE_VOLUME_OPTION,
            google_option=option_name,
        )
    )

FEATURE_SET_SPECS.extend(
    [
        feature_spec(
            "Model M - price + volume + GDELT + Reddit attention | zscore short",
            "price + volume + GDELT + Reddit attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT attention zscore short",
            reddit_option="Reddit attention zscore short",
        ),
        feature_spec(
            "Model M - price + volume + GDELT + Reddit attention | zscore medium",
            "price + volume + GDELT + Reddit attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT attention zscore medium",
            reddit_option="Reddit attention zscore medium",
        ),
        feature_spec(
            "Model N - price + volume + all attention | zscore short",
            "price + volume + all attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT attention zscore short",
            reddit_option="Reddit attention zscore short",
            google_option="Google score attention zscore 10d",
        ),
        feature_spec(
            "Model N - price + volume + all attention | zscore medium",
            "price + volume + all attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT attention zscore medium",
            reddit_option="Reddit attention zscore medium",
            google_option="Google score attention zscore 20d",
        ),
        feature_spec(
            "Model N - price + volume + all attention | percentile rank 20d",
            "price + volume + all attention",
            volume_option="volume percentile rank 20d",
            gdelt_option="GDELT attention percentile rank 20d",
            reddit_option="Reddit attention percentile rank 20d",
            google_option="Google score attention percentile rank 20d",
        ),
        feature_spec(
            "Model D - price + volume + GDELT + Reddit | zscore short",
            "price + volume + GDELT + Reddit",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT zscore short",
            reddit_option="Reddit zscore short",
        ),
        feature_spec(
            "Model D - price + volume + GDELT + Reddit | zscore medium",
            "price + volume + GDELT + Reddit",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT zscore medium",
            reddit_option="Reddit zscore medium",
        ),
        feature_spec(
            "Model D - price + volume + GDELT + Reddit | clipped zscore short",
            "price + volume + GDELT + Reddit",
            volume_option="volume zscore 20d clipped",
            gdelt_option="GDELT zscore short clipped",
            reddit_option="Reddit zscore short clipped",
        ),
        feature_spec(
            "Model H - price + volume + GDELT + Google | zscore short",
            "price + volume + GDELT + Google",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT zscore short",
            google_option="Google zscore 10d",
        ),
        feature_spec(
            "Model I - price + volume + Reddit + Google | zscore short",
            "price + volume + Reddit + Google",
            volume_option=BASE_VOLUME_OPTION,
            reddit_option="Reddit zscore short",
            google_option="Google zscore 10d",
        ),
        feature_spec(
            "Model F - price + volume + all alternative data | zscore short",
            "price + volume + all alternative data",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT zscore short",
            reddit_option="Reddit zscore short",
            google_option="Google zscore 10d",
        ),
    ]
)


FEATURE_SET_SPECS.extend(
    [
        feature_spec(
            "Model O - price + volume + GDELT sentiment + Reddit attention | zscore short",
            "price + volume + GDELT sentiment + Reddit attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT sentiment zscore short",
            reddit_option="Reddit attention zscore short",
        ),
        feature_spec(
            "Model O - price + volume + GDELT sentiment + Reddit attention | zscore medium",
            "price + volume + GDELT sentiment + Reddit attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT sentiment zscore medium",
            reddit_option="Reddit attention zscore medium",
        ),
        feature_spec(
            "Model O - price + volume + GDELT sentiment + Reddit attention | clipped sentiment + short attention",
            "price + volume + GDELT sentiment + Reddit attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT sentiment zscore short clipped",
            reddit_option="Reddit attention zscore short",
        ),
        feature_spec(
            "Model P - price + volume + GDELT sentiment + Reddit + Google attention | zscore short",
            "price + volume + GDELT sentiment + Reddit + Google attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT sentiment zscore short",
            reddit_option="Reddit attention zscore short",
            google_option="Google score attention zscore 10d",
        ),
        feature_spec(
            "Model P - price + volume + GDELT sentiment + Reddit + Google attention | zscore medium",
            "price + volume + GDELT sentiment + Reddit + Google attention",
            volume_option=BASE_VOLUME_OPTION,
            gdelt_option="GDELT sentiment zscore medium",
            reddit_option="Reddit attention zscore medium",
            google_option="Google score attention zscore 20d",
        ),
    ]
)

def feature_option_columns(option_name: str, option_sources: list[dict[str, list[str]]]) -> list[str]:
    for source in option_sources:
        if option_name in source:
            return source[option_name]
    raise KeyError(f"Unknown feature option: {option_name}")


def features_from_spec(spec: dict) -> list[str]:
    features = list(PRICE_FEATURES)
    if spec["volume_option"] is not None:
        features.extend(VOLUME_FEATURE_OPTIONS[spec["volume_option"]])
    if spec["gdelt_option"] is not None:
        features.extend(feature_option_columns(spec["gdelt_option"], [GDELT_FEATURE_OPTIONS, GDELT_SENTIMENT_FEATURE_OPTIONS, GDELT_ATTENTION_FEATURE_OPTIONS]))
    if spec["reddit_option"] is not None:
        features.extend(feature_option_columns(spec["reddit_option"], [REDDIT_FEATURE_OPTIONS, REDDIT_ATTENTION_FEATURE_OPTIONS]))
    if spec["google_option"] is not None:
        features.extend(
            feature_option_columns(spec["google_option"], [GOOGLE_TRENDS_FEATURE_OPTIONS, GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS])
        )
    return features


FEATURE_SETS = {}
FEATURE_SET_METADATA = {}
SKIPPED_FEATURE_SETS = []

for spec in FEATURE_SET_SPECS:
    features = features_from_spec(spec)
    candidate = {**spec, "n_features": len(features), "features": features}
    if len(features) > CONFIG["max_features_per_model"]:
        SKIPPED_FEATURE_SETS.append(candidate)
        continue
    FEATURE_SETS[spec["feature_set"]] = features
    FEATURE_SET_METADATA[spec["feature_set"]] = candidate

FEATURE_SETS_TO_TEST = list(FEATURE_SETS.keys())

pd.DataFrame(SVM_PARAM_GRID)


,param_set,kernel,C,class_weight,gamma
0,linear_C0p01_balanced,linear,0.01,balanced,NaN
1,linear_C0p1_balanced,linear,0.10,balanced,NaN
2,linear_C1_balanced,linear,1.00,balanced,NaN
3,linear_C10_balanced,linear,10.00,balanced,NaN
4,rbf_C0p1_gammascale_balanced,rbf,0.10,balanced,scale
5,rbf_C0p1_gamma0p001_balanced,rbf,0.10,balanced,0.001
6,rbf_C0p1_gamma0p01_balanced,rbf,0.10,balanced,0.01
7,rbf_C0p1_gamma0p1_balanced,rbf,0.10,balanced,0.1
8,rbf_C1_gammascale_balanced,rbf,1.00,balanced,scale
9,rbf_C1_gamma0p001_balanced,rbf,1.00,balanced,0.001


In [3]:
from __future__ import annotations


def transformed_source_series(
    frame: pd.DataFrame,
    source_col: str,
    value_transform: str | None = None,
) -> pd.Series:
    values = frame[source_col].astype(float)
    if value_transform is None:
        return values
    if value_transform == "log1p":
        return np.log1p(values.clip(lower=0.0))
    raise ValueError(f"Unsupported value_transform: {value_transform}")


def add_trailing_zscore(
    frame: pd.DataFrame,
    source_col: str,
    output_col: str,
    window: int,
    *,
    value_transform: str | None = None,
    clip_value: float | None = None,
) -> None:
    values = transformed_source_series(frame, source_col, value_transform)
    grouped = values.groupby(frame["ticker"])
    rolling_mean = grouped.transform(lambda s: s.shift(1).rolling(window).mean())
    rolling_std = grouped.transform(lambda s: s.shift(1).rolling(window).std())
    zscore = (values - rolling_mean) / rolling_std.replace(0.0, np.nan)
    if clip_value is not None:
        zscore = zscore.clip(lower=-clip_value, upper=clip_value)
    frame[output_col] = zscore


def add_trailing_percentile_rank(
    frame: pd.DataFrame,
    source_col: str,
    output_col: str,
    window: int,
    *,
    value_transform: str | None = None,
) -> None:
    values = transformed_source_series(frame, source_col, value_transform)

    def current_vs_history_rank(window_values: np.ndarray) -> float:
        current_value = window_values[-1]
        history = window_values[:-1]
        history = history[~np.isnan(history)]
        if np.isnan(current_value) or len(history) == 0:
            return np.nan
        return float(np.mean(history <= current_value))

    frame[output_col] = values.groupby(frame["ticker"]).transform(
        lambda s: s.rolling(window + 1, min_periods=window + 1).apply(
            current_vs_history_rank,
            raw=True,
        )
    )


def build_feature_frame(raw_df: pd.DataFrame, neutral_band: float) -> pd.DataFrame:
    frame = raw_df.copy().sort_values(["ticker", "date"]).reset_index(drop=True)
    frame["reddit_sentiment_missing"] = frame["comm_reddit_vader_mean"].isna().astype(float)
    frame["gdelt_sentiment_missing"] = frame["gdelt_sentiment_score"].isna().astype(float)
    frame["comm_reddit_posts"] = frame["comm_reddit_posts"].fillna(0.0)
    frame["gdelt_articles"] = frame["gdelt_articles"].fillna(0.0)

    price_group = frame.groupby("ticker")["stock_price"]
    frame["return_1d"] = price_group.pct_change(1)
    frame["return_5d"] = price_group.pct_change(5)
    frame["return_20d"] = price_group.pct_change(20)
    frame["rolling_volatility_20d"] = (
        frame.groupby("ticker")["return_1d"].transform(lambda s: s.shift(1).rolling(20).std())
    )

    for window in [10, 20, 60]:
        add_trailing_zscore(frame, "stock_volume", f"volume_zscore_{window}d", window)
    add_trailing_zscore(frame, "stock_volume", "volume_zscore_20d_clip3", 20, clip_value=3.0)
    add_trailing_zscore(frame, "stock_volume", "volume_log1p_zscore_20d", 20, value_transform="log1p")
    add_trailing_percentile_rank(frame, "stock_volume", "volume_rank_20d", 20)

    for window in [10, 20]:
        add_trailing_zscore(frame, "gdelt_sentiment_score", f"gdelt_sentiment_zscore_{window}d", window)
    for window in [6, 20]:
        add_trailing_zscore(frame, "gdelt_articles", f"gdelt_article_count_zscore_{window}d", window)
        add_trailing_zscore(
            frame,
            "gdelt_articles",
            f"gdelt_article_count_log1p_zscore_{window}d",
            window,
            value_transform="log1p",
        )
    add_trailing_zscore(frame, "gdelt_sentiment_score", "gdelt_sentiment_zscore_10d_clip3", 10, clip_value=3.0)
    add_trailing_zscore(frame, "gdelt_articles", "gdelt_article_count_zscore_6d_clip3", 6, clip_value=3.0)
    add_trailing_percentile_rank(frame, "gdelt_sentiment_score", "gdelt_sentiment_rank_20d", 20)
    add_trailing_percentile_rank(frame, "gdelt_articles", "gdelt_article_count_rank_20d", 20)

    for window in [6, 20]:
        add_trailing_zscore(frame, "comm_reddit_vader_mean", f"reddit_sentiment_zscore_{window}d", window)
        add_trailing_zscore(frame, "comm_reddit_posts", f"reddit_comment_count_zscore_{window}d", window)
        add_trailing_zscore(
            frame,
            "comm_reddit_posts",
            f"reddit_comment_count_log1p_zscore_{window}d",
            window,
            value_transform="log1p",
        )
    add_trailing_zscore(frame, "comm_reddit_vader_mean", "reddit_sentiment_zscore_6d_clip3", 6, clip_value=3.0)
    add_trailing_zscore(frame, "comm_reddit_posts", "reddit_comment_count_zscore_6d_clip3", 6, clip_value=3.0)
    add_trailing_percentile_rank(frame, "comm_reddit_vader_mean", "reddit_sentiment_rank_20d", 20)
    add_trailing_percentile_rank(frame, "comm_reddit_posts", "reddit_comment_count_rank_20d", 20)

    for window in [10, 20, 60]:
        add_trailing_zscore(frame, "google_trends_score", f"google_trends_zscore_{window}d", window)
    add_trailing_zscore(frame, "google_trends_score", "google_trends_zscore_20d_clip3", 20, clip_value=3.0)
    add_trailing_percentile_rank(frame, "google_trends_score", "google_trends_rank_20d", 20)

    frame["future_return_1d"] = price_group.shift(-1) / frame["stock_price"] - 1.0
    frame["target"] = np.select(
        [
            frame["future_return_1d"] < -neutral_band,
            frame["future_return_1d"] > neutral_band,
        ],
        [0, 1],
        default=np.nan,
    )
    frame["target_available"] = frame["future_return_1d"].notna()
    frame["is_neutral"] = frame["target_available"] & frame["future_return_1d"].abs().le(neutral_band)
    return frame


def make_split_dates(
    frame: pd.DataFrame,
    test_size: float,
    validation_fraction_within_pretest: float,
    min_validation_dates: int,
    gap_days: int,
) -> tuple[list[pd.Timestamp], list[pd.Timestamp], list[pd.Timestamp]]:
    unique_dates = sorted(frame["date"].drop_duplicates())

    test_start_idx = int(np.floor(len(unique_dates) * (1.0 - test_size)))
    test_start_idx = min(max(test_start_idx, 2), len(unique_dates) - 1)
    pretest_end_idx = max(test_start_idx - gap_days, 1)
    pretest_dates = unique_dates[:pretest_end_idx]

    validation_size = int(np.floor(len(pretest_dates) * validation_fraction_within_pretest))
    validation_size = max(min_validation_dates, validation_size)
    validation_size = min(max(validation_size, 1), len(pretest_dates) - 1)

    validation_start_idx = len(pretest_dates) - validation_size
    train_end_idx = max(validation_start_idx - gap_days, 1)

    train_dates = unique_dates[:train_end_idx]
    validation_dates = pretest_dates[validation_start_idx:]
    test_dates = unique_dates[test_start_idx:]
    return train_dates, validation_dates, test_dates


def subset_by_dates(frame: pd.DataFrame, dates: list[pd.Timestamp]) -> pd.DataFrame:
    return frame[frame["date"].isin(dates)].copy()


def make_train_validation_test_split_by_date(
    frame: pd.DataFrame,
    test_size: float,
    validation_fraction_within_pretest: float,
    min_validation_dates: int,
    gap_days: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_dates, validation_dates, test_dates = make_split_dates(
        frame,
        test_size=test_size,
        validation_fraction_within_pretest=validation_fraction_within_pretest,
        min_validation_dates=min_validation_dates,
        gap_days=gap_days,
    )
    return subset_by_dates(frame, train_dates), subset_by_dates(frame, validation_dates), subset_by_dates(frame, test_dates)


def make_walk_forward_fold_specs(
    session_dates: list[pd.Timestamp],
    *,
    n_folds: int,
    validation_size: int,
    min_train_dates: int,
    gap_days: int,
) -> list[dict]:
    dates = sorted(pd.Series(session_dates).drop_duplicates())
    if len(dates) == 0:
        raise ValueError("No session dates available for walk-forward validation.")

    max_validation_size = (len(dates) - min_train_dates - gap_days) // n_folds
    if max_validation_size < 1:
        raise ValueError(
            "Not enough dates for requested walk-forward folds. "
            f"dates={len(dates)}, n_folds={n_folds}, min_train_dates={min_train_dates}, gap_days={gap_days}"
        )
    validation_size = min(validation_size, max_validation_size)
    first_validation_start = len(dates) - n_folds * validation_size

    fold_specs = []
    for fold_idx in range(n_folds):
        validation_start = first_validation_start + fold_idx * validation_size
        validation_end = validation_start + validation_size
        train_end = validation_start - gap_days
        train_fold_dates = dates[:train_end]
        validation_fold_dates = dates[validation_start:validation_end]
        if len(train_fold_dates) < min_train_dates or not validation_fold_dates:
            continue
        fold_specs.append(
            {
                "fold": fold_idx + 1,
                "train_dates": train_fold_dates,
                "validation_dates": validation_fold_dates,
                "train_date_min": min(train_fold_dates),
                "train_date_max": max(train_fold_dates),
                "validation_date_min": min(validation_fold_dates),
                "validation_date_max": max(validation_fold_dates),
                "train_n_dates": len(train_fold_dates),
                "validation_n_dates": len(validation_fold_dates),
            }
        )

    if len(fold_specs) != n_folds:
        raise ValueError(f"Created {len(fold_specs)} walk-forward folds, expected {n_folds}.")
    return fold_specs


def safe_auc(y_true: pd.Series, scores: np.ndarray) -> float:
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return float(roc_auc_score(y_true, scores))


def build_svm_pipeline_from_params(params: dict) -> Pipeline:
    svm_params = dict(params)
    svm_params.pop("param_set", None)
    if svm_params.get("kernel") == "linear":
        svm_params.pop("gamma", None)
    svm_params.setdefault("class_weight", "balanced")
    svm_params.setdefault("random_state", CONFIG["random_state"])
    svm_params["probability"] = False
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(**svm_params)),
        ]
    )


def add_google_trends_train_median_feature(
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_name = "google_trends_above_ticker_train_median"
    source_col = "google_trends_score"
    median_by_ticker = train_input_df.groupby("ticker")[source_col].median()
    fallback_median = train_input_df[source_col].median()

    def with_feature(frame: pd.DataFrame) -> pd.DataFrame:
        out = frame.copy()
        thresholds = out["ticker"].map(median_by_ticker)
        if pd.notna(fallback_median):
            thresholds = thresholds.fillna(fallback_median)
        out[feature_name] = np.where(
            out[source_col].notna() & thresholds.notna(),
            (out[source_col] > thresholds).astype(float),
            np.nan,
        )
        return out

    return with_feature(train_input_df), with_feature(eval_input_df)

In [4]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)
walk_forward_fold_specs = make_walk_forward_fold_specs(
    train_dates,
    n_folds=CONFIG["walk_forward_folds"],
    validation_size=CONFIG["walk_forward_validation_dates"],
    min_train_dates=CONFIG["walk_forward_min_train_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {
    "train": train_dates,
    "validation": validation_dates,
    "test": test_dates,
}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin([0.0, 1.0])].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)

split_summary_rows = []
for split_name in ["train", "validation", "test"]:
    all_split_df = feature_df[feature_df["split"].eq(split_name)]
    modeled_split_df = modeled_df[modeled_df["split"].eq(split_name)]
    split_summary_rows.append(
        {
            "split": split_name,
            "session_rows": len(all_split_df),
            "modeled_rows": len(modeled_split_df),
            "session_dates": all_split_df["date"].nunique(),
            "modeled_dates": modeled_split_df["date"].nunique(),
            "date_min": all_split_df["date"].min(),
            "date_max": all_split_df["date"].max(),
            "target_positive_rate": modeled_split_df["target"].mean(),
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
walk_forward_fold_summary_df = pd.DataFrame(
    [
        {
            "fold": spec["fold"],
            "train_n_dates": spec["train_n_dates"],
            "train_date_min": spec["train_date_min"],
            "train_date_max": spec["train_date_max"],
            "validation_n_dates": spec["validation_n_dates"],
            "validation_date_min": spec["validation_date_min"],
            "validation_date_max": spec["validation_date_max"],
        }
        for spec in walk_forward_fold_specs
    ]
)

split_summary_df

,split,session_rows,modeled_rows,session_dates,modeled_dates,date_min,date_max,target_positive_rate
0,train,5632,4456,704,704,2021-01-04,2023-10-19,0.511670
1,validation,1880,1424,235,235,2023-10-23,2024-09-27,0.568118
2,test,2512,1886,314,313,2024-10-01,2025-12-31,0.526511


In [5]:
walk_forward_fold_summary_df

,fold,train_n_dates,train_date_min,train_date_max,validation_n_dates,validation_date_min,validation_date_max
0,1,383,2021-01-04,2022-07-12,80,2022-07-14,2022-11-03
1,2,463,2021-01-04,2022-11-02,80,2022-11-04,2023-03-02
2,3,543,2021-01-04,2023-03-01,80,2023-03-03,2023-06-27
3,4,623,2021-01-04,2023-06-26,80,2023-06-28,2023-10-19


In [6]:
neutral_summary_by_ticker_df = (
    feature_df.groupby("ticker")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reset_index()
)
neutral_summary_by_ticker_df["neutral_rate_among_available"] = (
    neutral_summary_by_ticker_df["neutral"] / neutral_summary_by_ticker_df["target_available"]
)
neutral_summary_by_ticker_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_ticker_df["neutral_rate_among_available"]

neutral_summary_by_split_df = (
    feature_df[feature_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
neutral_summary_by_split_df["neutral_rate_among_available"] = (
    neutral_summary_by_split_df["neutral"] / neutral_summary_by_split_df["target_available"]
)
neutral_summary_by_split_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_split_df["neutral_rate_among_available"]

print("Neutral coverage by split")
print(neutral_summary_by_split_df.to_string(index=False))
print("\nNeutral coverage by ticker")
neutral_summary_by_ticker_df

Neutral coverage by split
     split  rows  target_available  neutral  neutral_rate_among_available  modeled_rate_among_available
     train  5632              5632     1176                      0.208807                      0.791193
validation  1880              1880      456                      0.242553                      0.757447
      test  2512              2504      618                      0.246805                      0.753195

Neutral coverage by ticker


,ticker,rows,target_available,neutral,neutral_rate_among_available,modeled_rate_among_available
0,AAPL,1255,1254,378,0.301435,0.698565
1,AMD,1255,1254,220,0.175439,0.824561
2,AMZN,1255,1254,300,0.239234,0.760766
3,GOOGL,1255,1254,319,0.254386,0.745614
4,META,1255,1254,278,0.221691,0.778309
5,MSFT,1255,1254,400,0.318979,0.681021
6,NVDA,1255,1254,173,0.137959,0.862041
7,TSLA,1255,1254,184,0.146730,0.853270


In [7]:
missing_requested_feature_sets = [name for name in FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f"Unknown feature sets: {missing_requested_feature_sets}")

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(features),
            "features": features,
        }
        for feature_set_name, features in FEATURE_SETS.items()
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

skipped_feature_sets_df = pd.DataFrame(SKIPPED_FEATURE_SETS)

print(f"Selection metric: {CONFIG['selection_metric']}")
print(f"Primary validation metric: {CONFIG['primary_validation_metric']}")
print(f"Walk-forward folds: {len(walk_forward_fold_specs)}")
print(f"Walk-forward stability penalty: {CONFIG['walk_forward_stability_penalty']}")
print(f"Tune decision threshold in each validation fold: {CONFIG['tune_decision_threshold']}")
print(f"Max features per model: {CONFIG['max_features_per_model']}")
print(f"Feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets: {sum('attention' in FEATURE_SET_METADATA[name]['feature_family'] for name in FEATURE_SETS_TO_TEST)}")
print(f"Skipped feature sets above max feature limit: {len(SKIPPED_FEATURE_SETS)}")
print(f"SVM parameter sets: {len(SVM_PARAM_GRID)}")
print(f"Walk-forward validation fits: {len(FEATURE_SETS_TO_TEST) * len(SVM_PARAM_GRID) * len(walk_forward_fold_specs)}")

candidate_feature_sets_df


Selection metric: selection_score
Primary validation metric: balanced_accuracy
Walk-forward folds: 4
Walk-forward stability penalty: 0.25
Tune decision threshold in each validation fold: True
Max features per model: 9
Feature sets to test: 53
Attention feature sets: 23
Skipped feature sets above max feature limit: 1
SVM parameter sets: 16
Walk-forward validation fits: 3392


,feature_set,feature_family,n_features,features
0,Model B - price + volume | volume log1p zscore...,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
1,Model B - price + volume | volume percentile r...,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
2,Model B - price + volume | volume zscore 10d,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
3,Model B - price + volume | volume zscore 20d,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
4,Model B - price + volume | volume zscore 20d c...,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
5,Model B - price + volume | volume zscore 60d,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
6,Model C - price + volume + GDELT | GDELT perce...,price + volume + GDELT,7,"[return_1d, return_5d, return_20d, rolling_vol..."
7,Model C - price + volume + GDELT | GDELT senti...,price + volume + GDELT,7,"[return_1d, return_5d, return_20d, rolling_vol..."
8,Model C - price + volume + GDELT | GDELT zscor...,price + volume + GDELT,7,"[return_1d, return_5d, return_20d, rolling_vol..."
9,Model C - price + volume + GDELT | GDELT zscor...,price + volume + GDELT,7,"[return_1d, return_5d, return_20d, rolling_vol..."


In [8]:
from __future__ import annotations


def prepare_train_eval_feature_frames(
    features: list[str],
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "google_trends_above_ticker_train_median" in features:
        return add_google_trends_train_median_feature(train_input_df, eval_input_df)
    return train_input_df, eval_input_df


def params_from_result_row(row: dict | pd.Series) -> dict:
    params = {
        "param_set": row["param_set"],
        "kernel": row["kernel"],
        "C": row["C"],
        "gamma": row.get("gamma", None),
        "class_weight": row["class_weight"],
    }
    if pd.isna(params["gamma"]):
        params.pop("gamma")
    return params


def candidate_thresholds_from_scores(scores: np.ndarray) -> np.ndarray:
    finite_scores = np.asarray(scores, dtype=float)
    finite_scores = finite_scores[np.isfinite(finite_scores)]
    if len(finite_scores) == 0:
        return np.array([0.0])
    quantiles = np.linspace(
        CONFIG["threshold_min_quantile"],
        CONFIG["threshold_max_quantile"],
        CONFIG["threshold_grid_size"],
    )
    thresholds = np.quantile(finite_scores, quantiles)
    return np.unique(np.r_[thresholds, 0.0])


def best_threshold_for_balanced_accuracy(y_true: pd.Series, scores: np.ndarray) -> tuple[float, float]:
    rows = []
    for threshold in candidate_thresholds_from_scores(scores):
        preds = (scores >= threshold).astype(int)
        rows.append((float(threshold), float(balanced_accuracy_score(y_true, preds))))
    return max(rows, key=lambda item: item[1])


def metrics_from_scores(y_true: pd.Series, scores: np.ndarray, threshold: float) -> dict:
    preds = (scores >= threshold).astype(int)
    return {
        "preds": preds,
        "predicted_positive_rate": float(np.mean(preds)),
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, preds)),
        "auc": safe_auc(y_true, scores),
    }


def evaluate_svm_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    decision_threshold: float | None = None,
    tune_threshold: bool = False,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    train_features_df, eval_features_df = prepare_train_eval_feature_frames(
        features,
        train_input_df,
        eval_input_df,
    )

    pipeline = build_svm_pipeline_from_params(params)
    pipeline.fit(train_features_df[features], train_features_df["target"])
    scores = pipeline.decision_function(eval_features_df[features])

    threshold_source = "fixed"
    if tune_threshold:
        decision_threshold, threshold_selection_balanced_accuracy = best_threshold_for_balanced_accuracy(
            eval_features_df["target"],
            scores,
        )
        threshold_source = "validation_tuned"
    else:
        threshold_selection_balanced_accuracy = np.nan
        if decision_threshold is None:
            decision_threshold = 0.0
            threshold_source = "default_zero"

    metric_result = metrics_from_scores(eval_features_df["target"], scores, float(decision_threshold))
    preds = metric_result.pop("preds")
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    row = {
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "volume_transform": metadata.get("volume_option"),
        "gdelt_transform": metadata.get("gdelt_option"),
        "reddit_transform": metadata.get("reddit_option"),
        "google_transform": metadata.get("google_option"),
        "features": ", ".join(features),
        "param_set": params["param_set"],
        "kernel": params.get("kernel"),
        "C": params.get("C"),
        "gamma": params.get("gamma", None),
        "class_weight": params.get("class_weight"),
        "n_features": len(features),
        "train_rows": len(train_input_df),
        "eval_rows": len(eval_input_df),
        "eval_positive_rate": float(eval_features_df["target"].mean()),
        "decision_threshold": float(decision_threshold),
        "threshold_source": threshold_source,
        "threshold_selection_balanced_accuracy": threshold_selection_balanced_accuracy,
        **metric_result,
        "n_support_vectors": int(pipeline.named_steps["model"].n_support_.sum()),
    }

    if not return_predictions:
        return row

    predictions_df = eval_features_df[["date", "ticker", "target"]].copy()
    predictions_df["score"] = scores
    predictions_df["prediction"] = preds
    predictions_df["decision_threshold"] = float(decision_threshold)
    return row, predictions_df


def evaluate_svm_config_walk_forward(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    modeled_input_df: pd.DataFrame,
    fold_specs: list[dict],
) -> tuple[dict, list[dict]]:
    fold_rows = []
    for spec in fold_specs:
        fold_row = evaluate_svm_params(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            train_input_df=subset_by_dates(modeled_input_df, spec["train_dates"]),
            eval_input_df=subset_by_dates(modeled_input_df, spec["validation_dates"]),
            split_name="walk_forward_validation",
            tune_threshold=CONFIG["tune_decision_threshold"],
        )
        fold_rows.append(
            {
                **fold_row,
                "fold": spec["fold"],
                "fold_train_n_dates": spec["train_n_dates"],
                "fold_validation_n_dates": spec["validation_n_dates"],
                "fold_train_date_min": spec["train_date_min"],
                "fold_train_date_max": spec["train_date_max"],
                "fold_validation_date_min": spec["validation_date_min"],
                "fold_validation_date_max": spec["validation_date_max"],
            }
        )

    fold_results_df = pd.DataFrame(fold_rows)
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    primary_metric = CONFIG["primary_validation_metric"]
    metric_mean = float(fold_results_df[primary_metric].mean())
    metric_std = float(fold_results_df[primary_metric].std(ddof=0))
    selection_score = metric_mean - CONFIG["walk_forward_stability_penalty"] * metric_std

    summary_row = {
        "split": "walk_forward_validation",
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "volume_transform": metadata.get("volume_option"),
        "gdelt_transform": metadata.get("gdelt_option"),
        "reddit_transform": metadata.get("reddit_option"),
        "google_transform": metadata.get("google_option"),
        "features": ", ".join(features),
        "param_set": params["param_set"],
        "kernel": params.get("kernel"),
        "C": params.get("C"),
        "gamma": params.get("gamma", None),
        "class_weight": params.get("class_weight"),
        "n_features": len(features),
        "n_folds": len(fold_results_df),
        "selection_score": selection_score,
        "balanced_accuracy": metric_mean,
        "balanced_accuracy_std": metric_std,
        "balanced_accuracy_min": float(fold_results_df["balanced_accuracy"].min()),
        "accuracy": float(fold_results_df["accuracy"].mean()),
        "accuracy_std": float(fold_results_df["accuracy"].std(ddof=0)),
        "auc": float(fold_results_df["auc"].mean()),
        "auc_std": float(fold_results_df["auc"].std(ddof=0)),
        "predicted_positive_rate": float(fold_results_df["predicted_positive_rate"].mean()),
        "decision_threshold": float(fold_results_df["decision_threshold"].median()),
        "threshold_source": "walk_forward_fold_median",
        "n_support_vectors": float(fold_results_df["n_support_vectors"].mean()),
    }
    return summary_row, fold_rows


def metric_from_prediction_frame(prediction_frame: pd.DataFrame, metric_name: str) -> float:
    if metric_name == "accuracy":
        return float(accuracy_score(prediction_frame["target"], prediction_frame["prediction"]))
    if metric_name == "balanced_accuracy":
        return float(balanced_accuracy_score(prediction_frame["target"], prediction_frame["prediction"]))
    if metric_name == "auc":
        return safe_auc(prediction_frame["target"], prediction_frame["score"].to_numpy())
    raise ValueError(f"Unsupported metric for bootstrap: {metric_name}")


def block_bootstrap_ci_by_date(
    prediction_frame: pd.DataFrame,
    metric_names: list[str],
    *,
    n_bootstrap: int,
    ci: float,
    random_state: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    grouped_by_date = {date: group for date, group in prediction_frame.groupby("date")}
    dates = np.array(list(grouped_by_date.keys()), dtype=object)
    alpha = (1.0 - ci) / 2.0
    rows = []

    for metric_name in metric_names:
        samples = []
        for _ in range(n_bootstrap):
            sampled_dates = rng.choice(dates, size=len(dates), replace=True)
            sampled_frame = pd.concat([grouped_by_date[date] for date in sampled_dates], ignore_index=True)
            metric_value = metric_from_prediction_frame(sampled_frame, metric_name)
            if pd.notna(metric_value):
                samples.append(metric_value)
        rows.append(
            {
                "metric": metric_name,
                "estimate": metric_from_prediction_frame(prediction_frame, metric_name),
                "ci_lower": float(np.quantile(samples, alpha)) if samples else np.nan,
                "ci_upper": float(np.quantile(samples, 1.0 - alpha)) if samples else np.nan,
                "bootstrap_iterations": n_bootstrap,
                "block_unit": "date",
            }
        )

    return pd.DataFrame(rows)

In [9]:
selection_metric = CONFIG["selection_metric"]
walk_forward_grid_rows = []
walk_forward_fold_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in SVM_PARAM_GRID:
        summary_row, fold_rows = evaluate_svm_config_walk_forward(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            modeled_input_df=modeled_df,
            fold_specs=walk_forward_fold_specs,
        )
        walk_forward_grid_rows.append(summary_row)
        walk_forward_fold_rows.extend(fold_rows)

walk_forward_grid_results_df = pd.DataFrame(walk_forward_grid_rows)
walk_forward_fold_results_df = pd.DataFrame(walk_forward_fold_rows)
if selection_metric not in walk_forward_grid_results_df.columns:
    raise KeyError(f"Selection metric is not available: {selection_metric}")

validation_grid_results_df = walk_forward_grid_results_df.sort_values(
    [selection_metric, "balanced_accuracy", "auc", "accuracy", "feature_set", "param_set"],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)

validation_grid_results_df

,split,feature_set,feature_family,volume_transform,gdelt_transform,reddit_transform,google_transform,features,param_set,kernel,C,gamma,class_weight,n_features,n_folds,selection_score,balanced_accuracy,balanced_accuracy_std,balanced_accuracy_min,accuracy,accuracy_std,auc,auc_std,predicted_positive_rate,decision_threshold,threshold_source,n_support_vectors
0,walk_forward_validation,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,volume zscore 20d,GDELT attention zscore medium,NaN,NaN,"return_1d, return_5d, return_20d, rolling_vola...",linear_C0p1_balanced,linear,0.1,None,balanced,6,4,0.544687,0.551866,0.028717,0.513683,0.559601,0.020254,0.525648,0.045892,0.381778,0.467714,walk_forward_fold_median,3095.50
1,walk_forward_validation,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,volume zscore 20d,GDELT attention zscore medium,NaN,NaN,"return_1d, return_5d, return_20d, rolling_vola...",linear_C10_balanced,linear,10.0,None,balanced,6,4,0.542884,0.549648,0.027055,0.513705,0.539056,0.040631,0.525543,0.045710,0.553824,0.258197,walk_forward_fold_median,3092.75
2,walk_forward_validation,Model N - price + volume + all attention | zsc...,price + volume + all attention,volume zscore 20d,GDELT attention zscore medium,Reddit attention zscore medium,Google score attention zscore 20d,"return_1d, return_5d, return_20d, rolling_vola...",rbf_C1_gamma0p01_balanced,rbf,1.0,0.01,balanced,8,4,0.542789,0.548708,0.023676,0.516434,0.532027,0.038227,0.521534,0.042007,0.517384,0.297857,walk_forward_fold_median,3102.00
3,walk_forward_validation,Model N - price + volume + all attention | zsc...,price + volume + all attention,volume zscore 20d,GDELT attention zscore medium,Reddit attention zscore medium,Google score attention zscore 20d,"return_1d, return_5d, return_20d, rolling_vola...",rbf_C10_gamma0p001_balanced,rbf,10.0,0.001,balanced,8,4,0.542728,0.549304,0.026303,0.513705,0.538073,0.039749,0.523429,0.044937,0.544714,0.248326,walk_forward_fold_median,3105.25
4,walk_forward_validation,Model M - price + volume + GDELT + Reddit atte...,price + volume + GDELT + Reddit attention,volume zscore 20d,GDELT attention zscore medium,Reddit attention zscore medium,NaN,"return_1d, return_5d, return_20d, rolling_vola...",linear_C0p1_balanced,linear,0.1,None,balanced,7,4,0.542437,0.550219,0.031131,0.506519,0.559601,0.020254,0.524724,0.047823,0.354250,0.467644,walk_forward_fold_median,3096.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
843,walk_forward_validation,Model E - price + volume + Reddit | Reddit zsc...,price + volume + Reddit,volume zscore 20d,NaN,Reddit zscore short clipped,NaN,"return_1d, return_5d, return_20d, rolling_vola...",rbf_C10_gamma0p1_balanced,rbf,10.0,0.1,balanced,7,4,0.514663,0.517007,0.009375,0.507308,0.518508,0.019703,0.489448,0.009921,0.511282,0.062691,walk_forward_fold_median,2858.25
844,walk_forward_validation,Model E - price + volume + Reddit | Reddit sen...,price + volume + Reddit,volume zscore 20d,NaN,Reddit sentiment zscore + log comments,NaN,"return_1d, return_5d, return_20d, rolling_vola...",rbf_C10_gamma0p1_balanced,rbf,10.0,0.1,balanced,7,4,0.513881,0.516356,0.009899,0.508349,0.492755,0.028357,0.484283,0.015102,0.502351,-0.074043,walk_forward_fold_median,2866.00
845,walk_forward_validation,Model E - price + volume + Reddit | Reddit zsc...,price + volume + Reddit,volume zscore 20d,NaN,Reddit zscore short,NaN,"return_1d, return_5d, return_20d, rolling_vola...",rbf_C1_gammascale_balanced,rbf,1.0,scale,balanced,7,4,0.513352,0.516118,0.011066,0.502218,0.526300,0.030062,0.489843,0.013194,0.786030,-0.636422,walk_forward_fold_median,3009.50
846,walk_forward_validation,Model E - price + volume + Reddit | Reddit zsc...,price + volume + Reddit,volume zscore 20d,NaN,Reddit zscore short clipped,NaN,"return_1d, return_5d, return_20d, rolling_vola...",rbf_C1_gammascale_balanced,rbf,1.0,scale,balanced,7,4,0.513022,

In [10]:
validation_best_by_feature_set_df = (
    validation_grid_results_df.sort_values(
        ["feature_set", CONFIG["selection_metric"], "balanced_accuracy", "auc", "accuracy"],
        ascending=[True, False, False, False, False],
    )
    .groupby("feature_set", as_index=False)
    .head(1)
    .sort_values([CONFIG["selection_metric"], "balanced_accuracy", "auc", "accuracy"], ascending=False)
    .reset_index(drop=True)
)

validation_best_by_feature_set_display_df = validation_best_by_feature_set_df.rename(
    columns={
        "param_set": "best_param_set_by_walk_forward_score",
        "balanced_accuracy": "walk_forward_mean_balanced_accuracy",
        "balanced_accuracy_std": "walk_forward_std_balanced_accuracy",
        "accuracy": "walk_forward_mean_accuracy",
        "auc": "walk_forward_mean_auc",
    }
)

validation_best_by_feature_set_display_df[
    [
        "feature_set",
        "feature_family",
        "n_features",
        "best_param_set_by_walk_forward_score",
        "kernel",
        "C",
        "gamma",
        "selection_score",
        "walk_forward_mean_balanced_accuracy",
        "walk_forward_std_balanced_accuracy",
        "walk_forward_mean_accuracy",
        "walk_forward_mean_auc",
        "predicted_positive_rate",
    ]
]

,feature_set,feature_family,n_features,best_param_set_by_walk_forward_score,kernel,C,gamma,selection_score,walk_forward_mean_balanced_accuracy,walk_forward_std_balanced_accuracy,walk_forward_mean_accuracy,walk_forward_mean_auc,predicted_positive_rate
0,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,6,linear_C0p1_balanced,linear,0.10,None,0.544687,0.551866,0.028717,0.559601,0.525648,0.381778
1,Model N - price + volume + all attention | zsc...,price + volume + all attention,8,rbf_C1_gamma0p01_balanced,rbf,1.00,0.01,0.542789,0.548708,0.023676,0.532027,0.521534,0.517384
2,Model M - price + volume + GDELT + Reddit atte...,price + volume + GDELT + Reddit attention,7,linear_C0p1_balanced,linear,0.10,None,0.542437,0.550219,0.031131,0.559601,0.524724,0.354250
3,Model C - price + volume + GDELT | GDELT perce...,price + volume + GDELT,7,rbf_C10_gamma0p1_balanced,rbf,10.00,0.1,0.541967,0.544251,0.009138,0.532181,0.526808,0.294026
4,Model D - price + volume + GDELT + Reddit | zs...,price + volume + GDELT + Reddit,9,rbf_C10_gamma0p001_balanced,rbf,10.00,0.001,0.541935,0.546561,0.018504,0.539344,0.519066,0.542858
5,Model C - price + volume + GDELT | GDELT zscor...,price + volume + GDELT,7,linear_C0p01_balanced,linear,0.01,None,0.541787,0.548074,0.025148,0.538489,0.524052,0.567697
6,Model D - price + volume + GDELT + Reddit | zs...,price + volume + GDELT + Reddit,9,linear_C0p1_balanced,linear,0.10,None,0.541629,0.547561,0.023725,0.538946,0.521676,0.570425
7,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,6,linear_C1_balanced,linear,1.00,None,0.540538,0.546205,0.022669,0.555941,0.524265,0.401160
8,Model N - price + volume + all attention | zsc...,price + volume + all attention,8,rbf_C10_gamma0p001_balanced,rbf,10.00,0.001,0.540527,0.546292,0.023058,0.546055,0.522715,0.283806
9,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,6,rbf_C10_gamma0p1_balanced,rbf,10.00,0.1,0.540416,0.544420,0.016016,0.532689,0.517027,0.487332


In [11]:
best_validation_params_df = validation_best_by_feature_set_df.copy()
final_selected_model_df = best_validation_params_df.head(1).copy()

final_selected_model_df[
    [
        "feature_set",
        "feature_family",
        "n_features",
        "param_set",
        "kernel",
        "C",
        "gamma",
        "selection_score",
        "balanced_accuracy",
        "balanced_accuracy_std",
        "balanced_accuracy_min",
        "accuracy",
        "auc",
        "predicted_positive_rate",
        "features",
    ]
]

,feature_set,feature_family,n_features,param_set,kernel,C,gamma,selection_score,balanced_accuracy,balanced_accuracy_std,balanced_accuracy_min,accuracy,auc,predicted_positive_rate,features
0,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,6,linear_C0p1_balanced,linear,0.1,None,0.544687,0.551866,0.028717,0.513683,0.559601,0.525648,0.381778,"return_1d, return_5d, return_20d, rolling_vola..."


In [12]:
final_selected_walk_forward_row = final_selected_model_df.iloc[0]
final_selected_params = params_from_result_row(final_selected_walk_forward_row)
final_selected_feature_set = final_selected_walk_forward_row["feature_set"]

final_threshold_calibration_result = evaluate_svm_params(
    feature_set_name=final_selected_feature_set,
    features=FEATURE_SETS[final_selected_feature_set],
    params=final_selected_params,
    train_input_df=train_df,
    eval_input_df=validation_df,
    split_name="validation_threshold_calibration",
    tune_threshold=True,
)
final_threshold_calibration_result_df = pd.DataFrame([final_threshold_calibration_result])

final_test_result, final_test_predictions_df = evaluate_svm_params(
    feature_set_name=final_selected_feature_set,
    features=FEATURE_SETS[final_selected_feature_set],
    params=final_selected_params,
    train_input_df=train_df,
    eval_input_df=test_df,
    split_name="test_final_walk_forward_selected",
    decision_threshold=final_threshold_calibration_result["decision_threshold"],
    tune_threshold=False,
    return_predictions=True,
)
final_test_result.update(
    {
        "walk_forward_selection_score": final_selected_walk_forward_row["selection_score"],
        "walk_forward_mean_balanced_accuracy": final_selected_walk_forward_row["balanced_accuracy"],
        "walk_forward_std_balanced_accuracy": final_selected_walk_forward_row["balanced_accuracy_std"],
        "threshold_calibration_accuracy": final_threshold_calibration_result["accuracy"],
        "threshold_calibration_balanced_accuracy": final_threshold_calibration_result["balanced_accuracy"],
        "threshold_calibration_auc": final_threshold_calibration_result["auc"],
    }
)
final_test_result_df = pd.DataFrame([final_test_result])

final_test_metric_ci_df = block_bootstrap_ci_by_date(
    final_test_predictions_df,
    ["balanced_accuracy", "accuracy", "auc"],
    n_bootstrap=CONFIG["bootstrap_iterations"],
    ci=CONFIG["bootstrap_ci"],
    random_state=CONFIG["random_state"],
)

final_test_result_df[
    [
        "feature_set",
        "feature_family",
        "n_features",
        "param_set",
        "kernel",
        "C",
        "gamma",
        "walk_forward_selection_score",
        "walk_forward_mean_balanced_accuracy",
        "walk_forward_std_balanced_accuracy",
        "decision_threshold",
        "threshold_calibration_balanced_accuracy",
        "accuracy",
        "balanced_accuracy",
        "auc",
        "predicted_positive_rate",
    ]
]

,feature_set,feature_family,n_features,param_set,kernel,C,gamma,walk_forward_selection_score,walk_forward_mean_balanced_accuracy,walk_forward_std_balanced_accuracy,decision_threshold,threshold_calibration_balanced_accuracy,accuracy,balanced_accuracy,auc,predicted_positive_rate
0,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,6,linear_C0p1_balanced,linear,0.1,None,0.544687,0.551866,0.028717,0.233784,0.530688,0.518028,0.511622,0.514959,0.621421


In [13]:
final_test_metric_ci_df

,metric,estimate,ci_lower,ci_upper,bootstrap_iterations,block_unit
0,balanced_accuracy,0.511622,0.481725,0.542930,1000,date
1,accuracy,0.518028,0.483187,0.552320,1000,date
2,auc,0.514959,0.475051,0.552638,1000,date


In [14]:
threshold_calibration_rows = []
test_rows = []

for row in best_validation_params_df.to_dict(orient="records"):
    params = params_from_result_row(row)
    feature_set_name = row["feature_set"]
    calibration_row = evaluate_svm_params(
        feature_set_name=feature_set_name,
        features=FEATURE_SETS[feature_set_name],
        params=params,
        train_input_df=train_df,
        eval_input_df=validation_df,
        split_name="validation_threshold_calibration",
        tune_threshold=True,
    )
    threshold_calibration_rows.append(calibration_row)
    test_rows.append(
        evaluate_svm_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_df,
            eval_input_df=test_df,
            split_name="test_walk_forward_selected_threshold",
            decision_threshold=calibration_row["decision_threshold"],
            tune_threshold=False,
        )
    )

threshold_calibration_results_df = pd.DataFrame(threshold_calibration_rows)
test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ["balanced_accuracy", "auc", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

baseline_walk_forward_row = best_validation_params_df[best_validation_params_df["feature_set"].eq(BASELINE_FEATURE_SET)].iloc[0]
baseline_calibration_row = threshold_calibration_results_df[threshold_calibration_results_df["feature_set"].eq(BASELINE_FEATURE_SET)].iloc[0]
baseline_test_row = test_best_validation_params_df[test_best_validation_params_df["feature_set"].eq(BASELINE_FEATURE_SET)].iloc[0]

simple_hyperparameter_summary_df = best_validation_params_df.rename(
    columns={
        "selection_score": "walk_forward_selection_score",
        "balanced_accuracy": "walk_forward_mean_balanced_accuracy",
        "balanced_accuracy_std": "walk_forward_std_balanced_accuracy",
        "balanced_accuracy_min": "walk_forward_min_balanced_accuracy",
        "accuracy": "walk_forward_mean_accuracy",
        "auc": "walk_forward_mean_auc",
        "predicted_positive_rate": "walk_forward_mean_predicted_positive_rate",
    }
).merge(
    threshold_calibration_results_df[
        ["feature_set", "accuracy", "balanced_accuracy", "auc", "predicted_positive_rate", "decision_threshold"]
    ].rename(
        columns={
            "accuracy": "threshold_calibration_accuracy",
            "balanced_accuracy": "threshold_calibration_balanced_accuracy",
            "auc": "threshold_calibration_auc",
            "predicted_positive_rate": "threshold_calibration_predicted_positive_rate",
            "decision_threshold": "threshold_calibration_decision_threshold",
        }
    ),
    on="feature_set",
    how="left",
).merge(
    test_best_validation_params_df[
        ["feature_set", "accuracy", "balanced_accuracy", "auc", "predicted_positive_rate", "decision_threshold"]
    ].rename(
        columns={
            "accuracy": "test_accuracy",
            "balanced_accuracy": "test_balanced_accuracy",
            "auc": "test_auc",
            "predicted_positive_rate": "test_predicted_positive_rate",
            "decision_threshold": "test_decision_threshold",
        }
    ),
    on="feature_set",
    how="left",
)

simple_hyperparameter_summary_df["walk_forward_selection_score_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["walk_forward_selection_score"] - baseline_walk_forward_row["selection_score"]
)
simple_hyperparameter_summary_df["walk_forward_balanced_accuracy_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["walk_forward_mean_balanced_accuracy"] - baseline_walk_forward_row["balanced_accuracy"]
)
simple_hyperparameter_summary_df["threshold_calibration_balanced_accuracy_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["threshold_calibration_balanced_accuracy"] - baseline_calibration_row["balanced_accuracy"]
)
simple_hyperparameter_summary_df["test_accuracy_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["test_accuracy"] - baseline_test_row["accuracy"]
)
simple_hyperparameter_summary_df["test_balanced_accuracy_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["test_balanced_accuracy"] - baseline_test_row["balanced_accuracy"]
)
simple_hyperparameter_summary_df["test_auc_lift_vs_baseline"] = (
    simple_hyperparameter_summary_df["test_auc"] - baseline_test_row["auc"]
)

summary_columns = [
    "feature_set",
    "feature_family",
    "n_features",
    "param_set",
    "kernel",
    "C",
    "gamma",
    "walk_forward_selection_score",
    "walk_forward_mean_balanced_accuracy",
    "walk_forward_std_balanced_accuracy",
    "walk_forward_min_balanced_accuracy",
    "walk_forward_balanced_accuracy_lift_vs_baseline",
    "threshold_calibration_decision_threshold",
    "threshold_calibration_balanced_accuracy",
    "threshold_calibration_auc",
    "test_balanced_accuracy",
    "test_accuracy",
    "test_auc",
    "test_balanced_accuracy_lift_vs_baseline",
    "test_auc_lift_vs_baseline",
    "features",
]

# Appendix table: all feature-set winners are walk-forward selected; test sorting below is descriptive only.
simple_hyperparameter_summary_df[summary_columns]

,feature_set,feature_family,n_features,param_set,kernel,C,gamma,walk_forward_selection_score,walk_forward_mean_balanced_accuracy,walk_forward_std_balanced_accuracy,walk_forward_min_balanced_accuracy,walk_forward_balanced_accuracy_lift_vs_baseline,threshold_calibration_decision_threshold,threshold_calibration_balanced_accuracy,threshold_calibration_auc,test_balanced_accuracy,test_accuracy,test_auc,test_balanced_accuracy_lift_vs_baseline,test_auc_lift_vs_baseline,features
0,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,6,linear_C0p1_balanced,linear,0.10,None,0.544687,0.551866,0.028717,0.513683,0.016147,0.233784,0.530688,0.505595,0.511622,0.518028,0.514959,-0.014081,-0.012759,"return_1d, return_5d, return_20d, rolling_vola..."
1,Model N - price + volume + all attention | zsc...,price + volume + all attention,8,rbf_C1_gamma0p01_balanced,rbf,1.00,0.01,0.542789,0.548708,0.023676,0.516434,0.012989,-0.002255,0.527893,0.506340,0.503445,0.510604,0.516145,-0.022258,-0.011573,"return_1d, return_5d, return_20d, rolling_vola..."
2,Model M - price + volume + GDELT + Reddit atte...,price + volume + GDELT + Reddit attention,7,linear_C0p1_balanced,linear,0.10,None,0.542437,0.550219,0.031131,0.506519,0.014500,0.218107,0.529324,0.505665,0.509491,0.516437,0.514791,-0.016211,-0.012927,"return_1d, return_5d, return_20d, rolling_vola..."
3,Model C - price + volume + GDELT | GDELT perce...,price + volume + GDELT,7,rbf_C10_gamma0p1_balanced,rbf,10.00,0.1,0.541967,0.544251,0.009138,0.528789,0.008532,0.927841,0.505479,0.476069,0.517358,0.497349,0.539644,-0.008345,0.011927,"return_1d, return_5d, return_20d, rolling_vola..."
4,Model D - price + volume + GDELT + Reddit | zs...,price + volume + GDELT + Reddit,9,rbf_C10_gamma0p001_balanced,rbf,10.00,0.001,0.541935,0.546561,0.018504,0.518554,0.010842,0.175735,0.536413,0.513869,0.511006,0.517497,0.522328,-0.014697,-0.005389,"return_1d, return_5d, return_20d, rolling_vola..."
5,Model C - price + volume + GDELT | GDELT zscor...,price + volume + GDELT,7,linear_C0p01_balanced,linear,0.01,None,0.541787,0.548074,0.025148,0.512304,0.012355,0.225231,0.529975,0.506726,0.510946,0.518028,0.514732,-0.014757,-0.012986,"return_1d, return_5d, return_20d, rolling_vola..."
6,Model D - price + volume + GDELT + Reddit | zs...,price + volume + GDELT + Reddit,9,linear_C0p1_balanced,linear,0.10,None,0.541629,0.547561,0.023725,0.514424,0.011842,0.257115,0.530556,0.504927,0.515771,0.521209,0.514306,-0.009932,-0.013412,"return_1d, return_5d, return_20d, rolling_vola..."
7,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,6,linear_C1_balanced,linear,1.00,None,0.540538,0.546205,0.022669,0.515084,0.010487,0.193497,0.522985,0.507661,0.509032,0.517497,0.516217,-0.016670,-0.011500,"return_1d, return_5d, return_20d, rolling_vola..."
8,Model N - price + volume + all attention | zsc...,price + volume + all attention,8,rbf_C10_gamma0p001_balanced,rbf,10.00,0.001,0.540527,0.546292,0.023058,0.512355,0.010573,-0.022512,0.524752,0.509657,0.509114,0.521739,0.516592,-0.016589,-0.011126,"return_1d, return_5d, return_20d, rolling_vola..."
9,Model J - price + volume + GDELT attention | G...,price + volume + GDELT attention,6,rbf_C10_gamma0p1_balanced,rbf,10.00,0.1,0.540416,0.544420,0.016016,0.527301,0.008701,0.982786,0.503231,0.476429,0.518599,0.497349,0.543103,-0.007104,0.015385,"return_1d, return_5d, return_20d, rolling_vola..."


In [15]:
PRICE_VOLUME_BASELINE_FAMILY = "price + volume"

research_question_columns = [
    "comparison_group",
    "feature_family",
    "feature_set",
    "n_features",
    "param_set",
    "walk_forward_mean_balanced_accuracy",
    "threshold_calibration_balanced_accuracy",
    "test_accuracy",
    "test_balanced_accuracy",
    "test_auc",
    "test_accuracy_lift_vs_price_volume",
    "test_balanced_accuracy_lift_vs_price_volume",
    "test_auc_lift_vs_price_volume",
    "improves_test_accuracy_vs_price_volume",
    "improves_test_balanced_accuracy_vs_price_volume",
]

baseline_price_volume_row = (
    simple_hyperparameter_summary_df[
        simple_hyperparameter_summary_df["feature_family"].eq(PRICE_VOLUME_BASELINE_FAMILY)
    ]
    .sort_values(
        ["walk_forward_selection_score", "walk_forward_mean_balanced_accuracy", "threshold_calibration_balanced_accuracy"],
        ascending=False,
    )
    .head(1)
    .iloc[0]
)

alternative_data_comparison_df = simple_hyperparameter_summary_df.copy()
alternative_data_comparison_df["uses_alternative_data"] = ~alternative_data_comparison_df["feature_family"].isin(
    ["price only", "price + volume"]
)
alternative_data_comparison_df["comparison_baseline_feature_set"] = baseline_price_volume_row["feature_set"]
alternative_data_comparison_df["test_accuracy_lift_vs_price_volume"] = (
    alternative_data_comparison_df["test_accuracy"] - baseline_price_volume_row["test_accuracy"]
)
alternative_data_comparison_df["test_balanced_accuracy_lift_vs_price_volume"] = (
    alternative_data_comparison_df["test_balanced_accuracy"] - baseline_price_volume_row["test_balanced_accuracy"]
)
alternative_data_comparison_df["test_auc_lift_vs_price_volume"] = (
    alternative_data_comparison_df["test_auc"] - baseline_price_volume_row["test_auc"]
)
alternative_data_comparison_df["walk_forward_balanced_accuracy_lift_vs_price_volume"] = (
    alternative_data_comparison_df["walk_forward_mean_balanced_accuracy"]
    - baseline_price_volume_row["walk_forward_mean_balanced_accuracy"]
)
alternative_data_comparison_df["threshold_calibration_balanced_accuracy_lift_vs_price_volume"] = (
    alternative_data_comparison_df["threshold_calibration_balanced_accuracy"]
    - baseline_price_volume_row["threshold_calibration_balanced_accuracy"]
)

best_alternative_data_by_family_df = (
    alternative_data_comparison_df[alternative_data_comparison_df["uses_alternative_data"]]
    .sort_values(
        [
            "feature_family",
            "walk_forward_selection_score",
            "walk_forward_mean_balanced_accuracy",
            "threshold_calibration_balanced_accuracy",
            "test_balanced_accuracy",
        ],
        ascending=[True, False, False, False, False],
    )
    .groupby("feature_family", as_index=False)
    .head(1)
    .sort_values(
        ["test_accuracy_lift_vs_price_volume", "test_balanced_accuracy_lift_vs_price_volume", "test_auc_lift_vs_price_volume"],
        ascending=False,
    )
    .reset_index(drop=True)
)

baseline_research_question_df = pd.DataFrame(
    [
        {
            "comparison_group": "baseline",
            "feature_family": baseline_price_volume_row["feature_family"],
            "feature_set": baseline_price_volume_row["feature_set"],
            "n_features": baseline_price_volume_row["n_features"],
            "param_set": baseline_price_volume_row["param_set"],
            "walk_forward_mean_balanced_accuracy": baseline_price_volume_row["walk_forward_mean_balanced_accuracy"],
            "threshold_calibration_balanced_accuracy": baseline_price_volume_row["threshold_calibration_balanced_accuracy"],
            "test_accuracy": baseline_price_volume_row["test_accuracy"],
            "test_balanced_accuracy": baseline_price_volume_row["test_balanced_accuracy"],
            "test_auc": baseline_price_volume_row["test_auc"],
            "test_accuracy_lift_vs_price_volume": 0.0,
            "test_balanced_accuracy_lift_vs_price_volume": 0.0,
            "test_auc_lift_vs_price_volume": 0.0,
            "improves_test_accuracy_vs_price_volume": False,
            "improves_test_balanced_accuracy_vs_price_volume": False,
        }
    ]
)

alternative_research_question_df = best_alternative_data_by_family_df.copy()
alternative_research_question_df["comparison_group"] = "alternative data"
alternative_research_question_df["improves_test_accuracy_vs_price_volume"] = (
    alternative_research_question_df["test_accuracy_lift_vs_price_volume"] > 0
)
alternative_research_question_df["improves_test_balanced_accuracy_vs_price_volume"] = (
    alternative_research_question_df["test_balanced_accuracy_lift_vs_price_volume"] > 0
)

alternative_data_research_question_df = pd.concat(
    [
        baseline_research_question_df[research_question_columns],
        alternative_research_question_df[research_question_columns],
    ],
    ignore_index=True,
)

alternative_data_research_question_df

,comparison_group,feature_family,feature_set,n_features,param_set,walk_forward_mean_balanced_accuracy,threshold_calibration_balanced_accuracy,test_accuracy,test_balanced_accuracy,test_auc,test_accuracy_lift_vs_price_volume,test_balanced_accuracy_lift_vs_price_volume,test_auc_lift_vs_price_volume,improves_test_accuracy_vs_price_volume,improves_test_balanced_accuracy_vs_price_volume
0,baseline,price + volume,Model B - price + volume | volume percentile r...,5,rbf_C10_gamma0p001_balanced,0.540991,0.511826,0.522269,0.502061,0.527963,0.000000,0.000000,0.000000,False,False
1,alternative data,price + volume + GDELT sentiment + Reddit atte...,Model O - price + volume + GDELT sentiment + R...,7,rbf_C1_gamma0p01_balanced,0.542176,0.517005,0.531283,0.535036,0.535698,0.009014,0.032975,0.007735,True,True
2,alternative data,price + volume + Reddit attention,Model K - price + volume + Reddit attention | ...,6,rbf_C1_gamma0p01_balanced,0.539639,0.518469,0.523330,0.526751,0.532908,0.001060,0.024689,0.004945,True,True
3,alternative data,price + volume + Reddit,Model E - price + volume + Reddit | Reddit zsc...,7,rbf_C10_gamma0p001_balanced,0.539081,0.523018,0.521739,0.512046,0.530904,-0.000530,0.009984,0.002941,False,True
4,alternative data,price + volume + GDELT + Google,Model H - price + volume + GDELT + Google | zs...,8,rbf_C10_gamma0p001_balanced,0.542519,0.529112,0.519618,0.506197,0.513418,-0.002651,0.004136,-0.014545,False,True
5,alternative data,price + volume + GDELT attention,Model J - price + volume + GDELT attention | G...,6,linear_C0p1_balanced,0.551866,0.530688,0.518028,0.511622,0.514959,-0.004242,0.009561,-0.013005,False,True
6,alternative data,price + volume + GDELT + Reddit,Model D - price + volume + GDELT + Reddit | zs...,9,rbf_C10_gamma0p001_balanced,0.546561,0.536413,0.517497,0.511006,0.522328,-0.004772,0.008944,-0.005635,False,True
7,alternative data,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,8,rbf_C10_gamma0p001_balanced,0.539561,0.522471,0.516967,0.505710,0.524127,-0.005302,0.003648,-0.003836,False,True
8,alternative data,price + volume + GDELT + Reddit attention,Model M - price + volume + GDELT + Reddit atte...,7,linear_C0p1_balanced,0.550219,0.529324,0.516437,0.509491,0.514791,-0.005832,0.007430,-0.013173,False,True
9,alternative data,price + volume + Reddit + Google,Model I - price + volume + Reddit + Google | z...,8,rbf_C10_gamma0p001_balanced,0.537711,0.520704,0.514846,0.507586,0.524297,-0.007423,0.005525,-0.003666,False,True


In [16]:
attention_feature_summary_df = simple_hyperparameter_summary_df[
    simple_hyperparameter_summary_df["feature_family"].str.contains("attention", case=False, na=False)
].copy()

best_attention_by_family_df = (
    attention_feature_summary_df.sort_values(
        [
            "feature_family",
            "walk_forward_selection_score",
            "walk_forward_mean_balanced_accuracy",
            "threshold_calibration_balanced_accuracy",
            "test_balanced_accuracy",
        ],
        ascending=[True, False, False, False, False],
    )
    .groupby("feature_family", as_index=False)
    .head(1)
    .sort_values(
        ["test_accuracy_lift_vs_baseline", "test_balanced_accuracy_lift_vs_baseline", "test_auc_lift_vs_baseline"],
        ascending=False,
    )
    .reset_index(drop=True)
)

attention_research_question_df = best_attention_by_family_df[
    [
        "feature_family",
        "feature_set",
        "n_features",
        "param_set",
        "kernel",
        "C",
        "gamma",
        "walk_forward_mean_balanced_accuracy",
        "threshold_calibration_balanced_accuracy",
        "test_accuracy",
        "test_balanced_accuracy",
        "test_auc",
        "test_accuracy_lift_vs_baseline",
        "test_balanced_accuracy_lift_vs_baseline",
        "test_auc_lift_vs_baseline",
        "features",
    ]
]

attention_research_question_df


,feature_family,feature_set,n_features,param_set,kernel,C,gamma,walk_forward_mean_balanced_accuracy,threshold_calibration_balanced_accuracy,test_accuracy,test_balanced_accuracy,test_auc,test_accuracy_lift_vs_baseline,test_balanced_accuracy_lift_vs_baseline,test_auc_lift_vs_baseline,features
0,price + volume + GDELT sentiment + Reddit atte...,Model O - price + volume + GDELT sentiment + R...,7,rbf_C1_gamma0p01_balanced,rbf,1.0,0.01,0.542176,0.517005,0.531283,0.535036,0.535698,0.011135,0.009334,0.007981,"return_1d, return_5d, return_20d, rolling_vola..."
1,price + volume + Reddit attention,Model K - price + volume + Reddit attention | ...,6,rbf_C1_gamma0p01_balanced,rbf,1.0,0.01,0.539639,0.518469,0.523330,0.526751,0.532908,0.003181,0.001048,0.005191,"return_1d, return_5d, return_20d, rolling_vola..."
2,price + volume + GDELT attention,Model J - price + volume + GDELT attention | G...,6,linear_C0p1_balanced,linear,0.1,None,0.551866,0.530688,0.518028,0.511622,0.514959,-0.002121,-0.014081,-0.012759,"return_1d, return_5d, return_20d, rolling_vola..."
3,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,8,rbf_C10_gamma0p001_balanced,rbf,10.0,0.001,0.539561,0.522471,0.516967,0.505710,0.524127,-0.003181,-0.019993,-0.003591,"return_1d, return_5d, return_20d, rolling_vola..."
4,price + volume + GDELT + Reddit attention,Model M - price + volume + GDELT + Reddit atte...,7,linear_C0p1_balanced,linear,0.1,None,0.550219,0.529324,0.516437,0.509491,0.514791,-0.003712,-0.016211,-0.012927,"return_1d, return_5d, return_20d, rolling_vola..."
5,price + volume + all attention,Model N - price + volume + all attention | zsc...,8,rbf_C1_gamma0p01_balanced,rbf,1.0,0.01,0.548708,0.527893,0.510604,0.503445,0.516145,-0.009544,-0.022258,-0.011573,"return_1d, return_5d, return_20d, rolling_vola..."
6,price + volume + Google score attention,Model L - price + volume + Google score attent...,6,rbf_C10_gamma0p001_balanced,rbf,10.0,0.001,0.544508,0.521956,0.506893,0.492647,0.497542,-0.013256,-0.033056,-0.030175,"return_1d, return_5d, return_20d, rolling_vola..."


In [17]:
# Descriptive comparison: each row is walk-forward selected and uses its fixed holdout-validation threshold on test.
best_validation_selected_model_by_family_test_accuracy_df = (
    simple_hyperparameter_summary_df.sort_values(
        ["feature_family", "test_accuracy", "test_balanced_accuracy", "test_auc", "walk_forward_selection_score"],
        ascending=[True, False, False, False, False],
    )
    .groupby("feature_family", as_index=False)
    .head(1)
    .sort_values(["test_balanced_accuracy", "test_auc"], ascending=False)
    .reset_index(drop=True)
)

family_comparison_columns = [
    "feature_family",
    "feature_set",
    "test_accuracy",
    "test_balanced_accuracy",
    "test_auc",
    "features",
]

best_validation_selected_model_by_family_test_accuracy_df[family_comparison_columns]

,feature_family,feature_set,test_accuracy,test_balanced_accuracy,test_auc,features
0,price + volume + GDELT sentiment + Reddit atte...,Model O - price + volume + GDELT sentiment + R...,0.531283,0.535036,0.535698,"return_1d, return_5d, return_20d, rolling_vola..."
1,price + volume + Reddit attention,Model K - price + volume + Reddit attention | ...,0.538176,0.531038,0.527870,"return_1d, return_5d, return_20d, rolling_vola..."
2,price + volume,Model B - price + volume | volume zscore 20d c...,0.525451,0.526284,0.526741,"return_1d, return_5d, return_20d, rolling_vola..."
3,price + volume + Reddit,Model E - price + volume + Reddit | Reddit zsc...,0.530753,0.522579,0.528549,"return_1d, return_5d, return_20d, rolling_vola..."
4,price only,Model A - price only,0.493637,0.515864,0.541206,"return_1d, return_5d, return_20d, rolling_vola..."
5,price + volume + GDELT + Reddit,Model D - price + volume + GDELT + Reddit | zs...,0.521209,0.515771,0.514306,"return_1d, return_5d, return_20d, rolling_vola..."
6,price + volume + GDELT,Model C - price + volume + GDELT | GDELT zscor...,0.521209,0.513065,0.518434,"return_1d, return_5d, return_20d, rolling_vola..."
7,price + volume + GDELT attention,Model J - price + volume + GDELT attention | G...,0.518028,0.511622,0.514959,"return_1d, return_5d, return_20d, rolling_vola..."
8,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,0.522800,0.511587,0.522647,"return_1d, return_5d, return_20d, rolling_vola..."
9,price + volume + GDELT + Reddit attention,Model M - price + volume + GDELT + Reddit atte...,0.516437,0.509491,0.514791,"return_1d, return_5d, return_20d, rolling_vola..."
